# Encoder-Decoder FWI: STH Model Experiment

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import torch
import os

from sth_model import create_true_model, create_initial_model, create_acquisition, NX, NZ, DX
from network import EncoderDecoder
from denise_fwi import DeniseInterface
from train import train, prepare_shot_gathers

## True and Initial Models (Fig. 3)

In [ ]:
def plot_models(vp, vs, rho, title_prefix='', src=None, rec=None):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    extent = [0, NX*DX/1000, NZ*DX/1000, 0]
    for ax, arr, name in zip(axes, [vp, vs, rho], ['Vp (km/s)', 'Vs (km/s)', 'Rho (g/cm3)']):
        im = ax.imshow(arr, cmap='RdBu_r', extent=extent, aspect='auto')
        if src is not None:
            ax.scatter(src.x/1000, src.y/1000, 3, color='w')
        if rec is not None:
            ax.scatter(rec.x/1000, rec.y/1000, 1, color='m')
        ax.set_title(f'{title_prefix}{name}')
        ax.set_xlabel('x (km)'); ax.set_ylabel('z (km)')
        divider = make_axes_locatable(ax)
        cax = divider.append_axes('right', size='5%', pad=0.05)
        plt.colorbar(im, cax=cax)
    plt.tight_layout()

In [ ]:
true_model = create_true_model()
init_model = create_initial_model()
src, rec = create_acquisition()

plot_models(true_model.vp/1000, true_model.vs/1000, true_model.rho/1000, 'True ', src, rec)
plot_models(init_model.vp/1000, init_model.vs/1000, init_model.rho/1000, 'Initial ')

## Shot Gathers

In [ ]:
di = DeniseInterface(obs_folder='./outputs_sth_obs/', fwi_folder='./outputs_sth_fwi/', verbose=0)
shots_x = di.load_observed_shots(keys=['_x'])
shots_y = di.load_observed_shots(keys=['_y'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, shot, title in zip(axes, [shots_x[14], shots_y[14]], ['Vx shot 15', 'Vy shot 15']):
    vmax = 0.05 * np.max(np.abs(shot))
    ax.imshow(shot.T, cmap='Greys', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_title(title); ax.set_xlabel('Receiver'); ax.set_ylabel('Time sample')
plt.tight_layout()

## Run Training

In [ ]:
# Run training (each iteration ~25s, 350 iters ~ 2.5 hours)
# import argparse
# args = argparse.Namespace(n_iters=350, lr=0.0025, time_downsample=4,
#                           log_interval=10, save_interval=50, save_dir='checkpoints')
# train(args)

# Or load from saved checkpoint:
# history = np.load('checkpoints/history.npz')

## Convergence Curve (Fig. 5)

In [ ]:
history = np.load('checkpoints/history.npz')
iters = history['iter']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(iters, history['data_loss'] / max(history['data_loss'][0], 1e-20), 'o-', label='data loss')
ax1.set_xlabel('Iterations'); ax1.set_ylabel('Normalized Loss')
ax1.legend(); ax1.grid(True); ax1.set_title('Data Loss')

ax2.semilogy(iters, history['mse_vp'] / max(history['mse_vp'][0], 1e-20), label='Vp MSE')
ax2.semilogy(iters, history['mse_vs'] / max(history['mse_vs'][0], 1e-20), label='Vs MSE')
ax2.semilogy(iters, history['mse_rho'] / max(history['mse_rho'][0], 1e-20), label='Rho MSE')
ax2.set_xlabel('Iterations'); ax2.set_ylabel('Normalized MSE')
ax2.legend(); ax2.grid(True); ax2.set_title('Model Loss')

plt.tight_layout()

## Inverted Model (Fig. 4a)

In [ ]:
net = EncoderDecoder(28, NX, NZ, init_model.vp, init_model.vs, init_model.rho)
checkpoint = torch.load('checkpoints/final_model.pt', weights_only=True)
net.load_state_dict(checkpoint['model_state_dict'])

vx_tensor, vy_tensor = prepare_shot_gathers(di)

with torch.no_grad():
    vp, vs, rho = net(vx_tensor, vy_tensor)

plot_models(vp.squeeze().numpy()/1000, vs.squeeze().numpy()/1000, 
            rho.squeeze().numpy()/1000, 'Inverted ')

## Metrics (Tables II-IV)

In [ ]:
try:
    from skimage.metrics import structural_similarity as ssim
    _have_ssim = True
except ImportError:
    _have_ssim = False
    print("skimage not available; SSIM will be computed manually.")

def _ssim_manual(a, b, data_range):
    """Simple luminance-only approximation of SSIM."""
    mu_a, mu_b = a.mean(), b.mean()
    sigma_a = a.std(); sigma_b = b.std()
    sigma_ab = np.mean((a - mu_a) * (b - mu_b))
    c1 = (0.01 * data_range) ** 2
    c2 = (0.03 * data_range) ** 2
    return ((2*mu_a*mu_b + c1) * (2*sigma_ab + c2)) / ((mu_a**2 + mu_b**2 + c1) * (sigma_a**2 + sigma_b**2 + c2))

vp_inv = vp.squeeze().numpy()
vs_inv = vs.squeeze().numpy()
rho_inv = rho.squeeze().numpy()

for name, inv, true in [('Vp', vp_inv, true_model.vp), 
                          ('Vs', vs_inv, true_model.vs),
                          ('Rho', rho_inv, true_model.rho)]:
    mse = np.mean((inv - true)**2)
    mae = np.mean(np.abs(inv - true))
    data_range = true.max() - true.min()
    if _have_ssim:
        ssim_val = ssim(true, inv, data_range=data_range)
    else:
        ssim_val = _ssim_manual(inv, true, data_range)
    print(f'{name}: MSE={mse:.4f}, SSIM={ssim_val:.4f}, MAE={mae:.4f}')